In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create plots directory if it doesn't exist
if not os.path.exists('plots'):
    os.makedirs('plots')

In [31]:
# Function to create and save confusion matrices
def create_confusion_matrices(save_dir, site_names=None):
    """
    Create and save confusion matrices from saved site prediction data.
    
    Parameters
    ----------
    save_dir : str
        Directory where site prediction data is saved
    site_names : list, optional
        List of site names for axis labels
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import torch
    
    # Define directory with saved predictions
    data_dir = os.path.join(save_dir, 'site_predictions')
    
    if not os.path.exists(data_dir):
        print(f"No site prediction data found in {data_dir}")
        return
        
    # Create directory for confusion matrix plots
    plot_dir = os.path.join(save_dir, 'confusion_matrices')
    os.makedirs(plot_dir, exist_ok=True)
    
    # Get all saved files
    true_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.npy') and 'true_sites' in f])
    
    # Group files by prefix and epoch
    file_groups = {}
    for f in true_files:
        # Extract prefix and epoch from filename
        parts = f.split('_')
        prefix = parts[0]
        epoch = int(parts[-1].replace('.npy', ''))
        
        # Add to groups
        if (prefix, epoch) not in file_groups:
            file_groups[(prefix, epoch)] = {}
        
        file_groups[(prefix, epoch)]['true'] = os.path.join(data_dir, f)
        # Find corresponding predictions file
        pred_file = f.replace('true_sites', 'pred_sites')
        if os.path.exists(os.path.join(data_dir, pred_file)):
            file_groups[(prefix, epoch)]['pred'] = os.path.join(data_dir, pred_file)
    
    print(f"Found {len(file_groups)} pairs of true/predicted site data")
    
    # Create confusion matrices
    for (prefix, epoch), files in file_groups.items():
        if 'true' in files and 'pred' in files:
            # Load data
            true_sites = np.load(files['true'])
            pred_sites = np.load(files['pred'])
            
            # Compute confusion matrix
            cm = confusion_matrix(true_sites, pred_sites, normalize='true')
            
            # Create display with labels if provided
            if site_names is not None:
                labels = site_names
            else:
                num_sites = len(np.unique(np.concatenate([true_sites, pred_sites])))
                labels = [f"Site {i}" for i in range(num_sites)]
                
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
            
            # Create figure and plot
            fig, ax = plt.subplots(figsize=(10, 8))
            disp.plot(ax=ax, cmap='Blues', values_format='.2f')
            plt.title(f'{prefix.capitalize()} Confusion Matrix - Epoch {epoch}')
            
            # Save figure
            plt.tight_layout()
            plt.savefig(os.path.join(plot_dir, f'{prefix}_confusion_matrix_epoch_{epoch}.png'))
            plt.close()
            
            print(f"Created confusion matrix for {prefix} data at epoch {epoch}")
            
            # Also save a CSV of the confusion matrix for detailed analysis
            df = pd.DataFrame(cm, index=labels, columns=labels)
            df.to_csv(os.path.join(plot_dir, f'{prefix}_confusion_matrix_epoch_{epoch}.csv'))
            
    # Create a final summary visualization of confusion matrices over time
    create_confusion_matrix_summary(plot_dir, file_groups)
    
def create_confusion_matrix_summary(plot_dir, file_groups):
    """Create a summary visualization showing how confusion matrices evolve over time."""
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix
    
    # Only process validation data for the summary
    val_epochs = sorted([epoch for prefix, epoch in file_groups.keys() if prefix == 'val'])
    
    if len(val_epochs) <= 1:
        print("Not enough epochs for time series visualization")
        return
        
    # Calculate metrics over time
    epochs = []
    accuracies = []
    chance_divergences = []  # How far from chance (1/num_classes)
    
    for epoch in val_epochs:
        if ('val', epoch) in file_groups:
            files = file_groups[('val', epoch)]
            
            # Load data
            true_sites = np.load(files['true'])
            pred_sites = np.load(files['pred'])
            
            # Calculate metrics
            cm = confusion_matrix(true_sites, pred_sites, normalize='true')
            accuracy = np.trace(cm) / np.sum(cm)
            
            # Chance level is 1/num_classes
            num_classes = cm.shape[0]
            chance_level = 1.0 / num_classes
            
            # Calculate divergence from chance (positive: better than chance, negative: worse than chance)
            chance_divergence = accuracy - chance_level
            
            epochs.append(epoch)
            accuracies.append(accuracy)
            chance_divergences.append(chance_divergence)
    
    # Create the plot
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    color = 'tab:blue'
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy', color=color)
    ax1.plot(epochs, accuracies, color=color, marker='o', label='Accuracy')
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Add horizontal line for chance level
    chance_level = 1.0 / num_classes
    ax1.axhline(y=chance_level, color='gray', linestyle='--', label=f'Chance Level ({chance_level:.2f})')
    
    # Add second axis for divergence
    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel('Divergence from Chance', color=color)
    ax2.plot(epochs, chance_divergences, color=color, marker='x', label='Divergence from Chance')
    ax2.tick_params(axis='y', labelcolor=color)
    
    # Add horizontal line at zero divergence
    ax2.axhline(y=0, color='lightgray', linestyle=':')
    
    # Add combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')
    
    plt.title('Site Classification Performance Over Training')
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, 'site_performance_over_time.png'))
    plt.close()
    
    # Save the data as CSV for further analysis
    df = pd.DataFrame({
        'epoch': epochs,
        'accuracy': accuracies,
        'chance_level': [chance_level] * len(epochs),
        'divergence_from_chance': chance_divergences
    })
    df.to_csv(os.path.join(plot_dir, 'site_performance_over_time.csv'), index=False)
    print(f"Created site classification performance summary")

def save_site_predictions(true_sites, pred_sites, epoch, save_dir, prefix=''):
    """
    Save site prediction data to create confusion matrices.
    
    Parameters
    ----------
    true_sites : torch.Tensor
        Ground truth site labels
    pred_sites : torch.Tensor
        Predicted site labels
    epoch : int
        Current epoch number
    save_dir : str
        Directory to save the data
    prefix : str
        Prefix for saved filenames (e.g., 'train' or 'val')
    """
    import numpy as np
    import os
    
    # Convert tensors to numpy arrays if needed
    if isinstance(true_sites, torch.Tensor):
        true_sites = true_sites.cpu().numpy()
    if isinstance(pred_sites, torch.Tensor):
        pred_sites = pred_sites.cpu().numpy()
    
    # Save the data
    data_dir = os.path.join(save_dir, 'site_predictions')
    os.makedirs(data_dir, exist_ok=True)
    
    np.save(os.path.join(data_dir, f'{prefix}_true_sites_epoch_{epoch}.npy'), true_sites)
    np.save(os.path.join(data_dir, f'{prefix}_pred_sites_epoch_{epoch}.npy'), pred_sites)
    
    print(f"Saved {prefix} site prediction data for epoch {epoch}")

In [32]:
#AGE PREDICTOR PLOTS
sns.set_style('whitegrid')

# Read the CSV file
df = pd.read_csv('csv_files/age_predictor_metrics.csv')

# Plot 1: Training Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_loss'],color='blue', linewidth=2)
plt.title('Training Loss vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Loss', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/age_predictor_training_loss_plot.png', dpi=300)
plt.close()

# Plot 2: Validation Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['val_loss'], color='red', linewidth=2)
plt.title('Validation Loss vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Loss', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/age_predictor_validation_loss_plot.png', dpi=300)
plt.close()

# Plot R² Metrics vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_r2'], color='green', linewidth=2, label='Train R²')
plt.plot(df['epoch'], df['val_r2'], color='purple', linewidth=2, label='Validation R²')
plt.title('Age Predictor R² Metrics vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('R²', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/age_predictor_r2_plot.png', dpi=300)
plt.close()

print("Plots have been saved in the 'plots' directory.")

Plots have been saved in the 'plots' directory.


In [33]:
df = pd.read_csv('csv_files/site_predictor_metrics.csv')

# Plot 1: Training Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_loss'], color='blue', linewidth=2)
plt.title('Training Loss vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Loss', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/site_training_loss_plot.png', dpi=300)
plt.close()

# Plot 2: Validation Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['val_loss'], color='red', linewidth=2)
plt.title('Validation Loss vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Loss', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/site_validation_loss_plot.png', dpi=300)
plt.close()

# Plot 3: Training Accuracy vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_acc'], color='green', linewidth=2)
plt.title('Training Accuracy vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Accuracy', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/site_training_accuracy_plot.png', dpi=300)
plt.close()

# Plot 4: Validation Accuracy vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['val_acc'], color='purple', linewidth=2)
plt.title('Validation Accuracy vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/site_validation_accuracy_plot.png', dpi=300)
plt.close()

print("All four plots have been saved in the 'plots' directory.")

All four plots have been saved in the 'plots' directory.


In [34]:
df = pd.read_csv('csv_files/vae_metrics.csv')

# Plot 1: Training Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_loss'],  color='blue', linewidth=2)
plt.title('Training Loss vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Loss', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/vae_training_loss_plot.png', dpi=300)
plt.close()

# Plot 2: Validation Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['val_loss'],  color='red', linewidth=2)
plt.title('Validation Loss vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Loss', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/vae_validation_loss_plot.png', dpi=300)
plt.close()

# Plot 3: Training Reconstruction Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_recon_loss'],color='green', linewidth=2)
plt.title('Training Reconstruction Loss vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Reconstruction Loss', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/vae_training_recon_loss_plot.png', dpi=300)
plt.close()

# Plot 4: Validation Reconstruction Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['val_recon_loss'],  color='purple', linewidth=2)
plt.title('Validation Reconstruction Loss vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Reconstruction Loss', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/vae_validation_recon_loss_plot.png', dpi=300)
plt.close()

# Plot 5: Training KL Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_kl_loss'], color='orange', linewidth=2)
plt.title('Training KL Loss vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training KL Loss', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/vae_training_kl_loss_plot.png', dpi=300)
plt.close()

# Plot 6: Validation KL Loss vs Epoch
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['val_kl_loss'], color='brown', linewidth=2)
plt.title('Validation KL Loss vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation KL Loss', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/vae_validation_kl_loss_plot.png', dpi=300)
plt.close()

print("All six VAE metric plots have been saved in the 'plots' directory.")

All six VAE metric plots have been saved in the 'plots' directory.


In [35]:

df = pd.read_csv('csv_files/staged_combined_metrics.csv')

# Plot group 1: Overall losses
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_loss'], color='blue', linewidth=2, label='Train Loss')
plt.plot(df['epoch'], df['val_loss'], color='red', linewidth=2, label='Validation Loss')
plt.title('Overall Losses vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/combined_overall_loss_plot.png', dpi=300)
plt.close()

# Plot group 2: Reconstruction losses
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_recon_loss'], color='green', linewidth=2, label='Train Recon Loss')
plt.plot(df['epoch'], df['val_recon_loss'], color='purple', linewidth=2, label='Validation Recon Loss')
plt.title('Reconstruction Losses vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Reconstruction Loss', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/combined_recon_loss_plot.png', dpi=300)
plt.close()

# Plot group 3: KL losses
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_kl_loss'], color='orange', linewidth=2, label='Train KL Loss')
plt.plot(df['epoch'], df['val_kl_loss'], color='brown', linewidth=2, label='Validation KL Loss')
plt.title('KL Divergence Losses vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('KL Loss', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/combined_kl_loss_plot.png', dpi=300)
plt.close()

# Plot group 4: Age losses
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_age_loss'], color='skyblue', linewidth=2, label='Train Age Loss')
plt.plot(df['epoch'], df['val_age_loss'], color='navy', linewidth=2, label='Validation Age Loss')
plt.title('Age Prediction Losses vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Age Loss', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/combined_age_loss_plot.png', dpi=300)
plt.close()

# Plot group 5: Site losses
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_site_loss'], color='pink', linewidth=2, label='Train Site Loss')
plt.plot(df['epoch'], df['val_site_loss'], color='crimson', linewidth=2, label='Validation Site Loss')
plt.title('Site Prediction Losses vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Site Loss', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/combined_site_loss_plot.png', dpi=300)
plt.close()

# Plot group 6: Age MAE
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_age_mae'], color='cyan', linewidth=2, label='Train Age MAE')
plt.plot(df['epoch'], df['val_age_mae'], color='teal', linewidth=2, label='Validation Age MAE')
plt.title('Age Mean Absolute Error vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Age MAE', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/combined_age_mae_plot.png', dpi=300)
plt.close()

# Plot group 7: Site accuracy
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_site_acc'], color='lime', linewidth=2, label='Train Site Accuracy')
plt.plot(df['epoch'], df['val_site_acc'], color='darkgreen', linewidth=2, label='Validation Site Accuracy')
plt.title('Site Prediction Accuracy vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Site Accuracy', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/combined_site_accuracy_plot.png', dpi=300)
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_age_r2'], color='gold', linewidth=2, label='Train Age R²')
plt.plot(df['epoch'], df['val_age_r2'], color='darkgoldenrod', linewidth=2, label='Validation Age R²')
plt.title('Age Prediction R² vs Epoch', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('R²', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('plots/combined_age_r2_plot.png', dpi=300)
plt.close()

print("All seven combined metric plots have been saved in the 'plots' directory.")

All seven combined metric plots have been saved in the 'plots' directory.
